# 03. BPv7 심화: bounded node processing과 보안 경계

작성·검증 기준일: **2026-09-11**  
원문: [RFC 9171 — Bundle Protocol Version 7](https://www.rfc-editor.org/rfc/rfc9171.html)

## 학습 목표

1. reception → validation → dispatch → delivery/forwarding/deletion 흐름을 상태 전이로 관찰한다.
2. Previous Node, Bundle Age, Hop Count extension을 forwarding 시 갱신한다.
3. malformed input, 중복, 저장 공간 고갈, 긴 lifetime, 인증되지 않은 입력을 명시적 정책으로 다룬다.

> **중요:** 이것은 RFC 절차를 읽기 위한 in-memory **toy simulator**다. CBOR parser, CLA, 실제 route selection, persistent storage, status report, fragmentation queue, BPSec를 구현하지 않는다. 따라서 RFC 9171/9172 적합성이나 wire interoperability를 주장하지 않는다. 특히 아래 `authenticated` Boolean과 HMAC 예시는 trust-boundary 사고 실험일 뿐 BIB/BCB가 아니다.

관련 절: RFC 9171 §4.4, §5.3–5.7, §5.10–5.11, §7.2, §8. 실행은 **Run All**, 외부 패키지는 필요 없다.

## 1. 신뢰 경계 앞의 bounded input adapter

네트워크 바이트를 곧바로 객체로 신뢰하면 공격자가 거대한 길이, 잘못된 타입, 중복 extension으로 메모리와 제어 흐름을 흔들 수 있다. 실제 시스템에서는 먼저 **크기 제한이 있는 CBOR decoder**가 필요하다. 여기서는 그 다음 단계에서 받은 Python mapping이라고 가정하고 다음을 검사한다.

- 정확한 필드 집합과 타입(`bool`을 정수로 받지 않음)
- payload와 extension 개수 상한
- Previous Node(type 6), Bundle Age(type 7), Hop Count(type 10)의 중복
- hop limit 1..255와 음이 아닌 hop count/age

알 수 없는 extension을 무조건 거부하는 것은 이 toy의 보수적 정책이다. 실제 BPA는 block processing control flags에 따라 report/delete/discard/continue를 결정한다(§5.6).

In [1]:
from dataclasses import dataclass, replace
from enum import Enum
import hashlib
import hmac

class InputError(ValueError):
    pass

@dataclass(frozen=True)
class Extensions:
    previous_node: str | None = None
    bundle_age_ms: int | None = None
    hop_limit: int | None = None
    hop_count: int | None = None

@dataclass(frozen=True)
class ToyBundle:
    source: str
    destination: str
    creation_time_ms: int
    sequence_number: int
    lifetime_ms: int
    payload: bytes
    extensions: Extensions = Extensions()
    version: int = 7

    @property
    def identity(self):
        # 비-fragment bundle에 대한 identity 부분집합이다.
        return self.source, self.creation_time_ms, self.sequence_number

def parse_untrusted(raw, *, max_payload_bytes=4096, max_extensions=8) -> ToyBundle:
    """이미 CBOR에서 해석됐다고 가정한 mapping을 제한적으로 ToyBundle로 바꾼다."""
    if type(raw) is not dict:
        raise InputError("top-level 값은 dict여야 합니다")
    required = {"version", "source", "destination", "creation_timestamp",
                "lifetime_ms", "payload", "extension_blocks"}
    if set(raw) != required:
        raise InputError(f"필드 집합 불일치: {set(raw) ^ required}")
    if type(raw["version"]) is not int or raw["version"] != 7:
        raise InputError("version은 정수 7이어야 합니다")
    if not all(type(raw[name]) is str and raw[name] for name in ("source", "destination")):
        raise InputError("source/destination은 비어 있지 않은 문자열이어야 합니다")
    timestamp = raw["creation_timestamp"]
    if (type(timestamp) not in (list, tuple) or len(timestamp) != 2 or
            any(type(value) is not int or value < 0 for value in timestamp)):
        raise InputError("creation timestamp는 음이 아닌 정수 두 개여야 합니다")
    if type(raw["lifetime_ms"]) is not int or raw["lifetime_ms"] < 0:
        raise InputError("lifetime은 음이 아닌 int여야 합니다")
    payload = raw["payload"]
    if not isinstance(payload, bytes) or len(payload) > max_payload_bytes:
        raise InputError("payload 타입 또는 크기 제한 위반")
    blocks = raw["extension_blocks"]
    if type(blocks) is not list or len(blocks) > max_extensions:
        raise InputError("extension block 목록 타입 또는 개수 제한 위반")

    values: dict[int, object] = {}
    for block in blocks:
        if type(block) is not dict or set(block) != {"type", "value"}:
            raise InputError("extension은 type/value만 가진 dict여야 합니다")
        block_type = block["type"]
        if type(block_type) is not int or block_type not in (6, 7, 10):
            raise InputError(f"이 toy가 처리할 수 없는 extension type: {block_type!r}")
        if block_type in values:
            raise InputError(f"extension type {block_type} 중복")
        values[block_type] = block["value"]

    previous = values.get(6)
    if previous is not None and (type(previous) is not str or not previous):
        raise InputError("Previous Node 값은 비어 있지 않은 node ID 문자열이어야 합니다")
    age = values.get(7)
    if age is not None and (type(age) is not int or age < 0):
        raise InputError("Bundle Age는 음이 아닌 정수(ms)여야 합니다")
    hop = values.get(10)
    hop_limit = hop_count = None
    if hop is not None:
        if (type(hop) not in (list, tuple) or len(hop) != 2 or
                any(type(value) is not int for value in hop)):
            raise InputError("Hop Count 값은 [limit, count] 정수 쌍이어야 합니다")
        hop_limit, hop_count = hop
        if not 1 <= hop_limit <= 255 or hop_count < 0:
            raise InputError("hop limit/count 범위 위반")
    return ToyBundle(
        raw["source"], raw["destination"], timestamp[0], timestamp[1],
        raw["lifetime_ms"], payload,
        Extensions(previous, age, hop_limit, hop_count), raw["version"],
    )

In [2]:
raw_bundle = {
    "version": 7,
    "source": "dtn://earth",
    "destination": "dtn://mars/inbox",
    "creation_timestamp": [0, 7],  # 시계가 없으므로 Bundle Age가 필요하다.
    "lifetime_ms": 10_000,
    "payload": b"temperature=18",
    "extension_blocks": [
        {"type": 6, "value": "dtn://moon-relay"},
        {"type": 7, "value": 100},
        {"type": 10, "value": [2, 0]},
    ],
}
parsed = parse_untrusted(raw_bundle)
print(parsed)

ToyBundle(source='dtn://earth', destination='dtn://mars/inbox', creation_time_ms=0, sequence_number=7, lifetime_ms=10000, payload=b'temperature=18', extensions=Extensions(previous_node='dtn://moon-relay', bundle_age_ms=100, hop_limit=2, hop_count=0), version=7)


## 2. 상태 전이 simulator

상태 이름은 RFC의 절차를 이해하기 쉽게 축약했다. 규범 문서가 하나의 enum 상태 머신을 정의한다는 뜻은 아니다.

- reception 때 `Dispatch pending` retention constraint가 생긴다는 사실을 event에 기록한다.
- 목적지가 로컬이면 delivery, 아니면 route 유무에 따라 forwarding 또는 보관한다.
- forwarding 직전 Previous Node를 현 노드로 바꾸고 Bundle Age에 residence time을 더한다(§4.4, §5.4). 이 toy는 `SHOULD` 권고를 채택해 hop count도 1 증가시킨다.
- 이 toy는 `SHOULD` 권고를 채택해 **증가한 hop count가 limit을 초과할 때(`>`)** 삭제한다.
- lifetime도 age가 lifetime을 초과할 때(`>`) 만료다.

In [3]:
class State(str, Enum):
    RECEIVED = "received"
    VALIDATED = "validated"
    DISPATCHED = "dispatched"
    FORWARD_PENDING = "forward_pending"
    FORWARDED = "forwarded"
    DELIVERED = "delivered"
    DELETED = "deleted"

@dataclass(frozen=True)
class ProcessResult:
    state: State
    reason: str
    bundle: ToyBundle | None
    events: tuple[str, ...]

class ToyNode:
    def __init__(self, node_id: str, local_endpoints=(), *, max_payload_bytes=4096,
                 max_storage_bytes=8192, local_lifetime_override_ms=None,
                 require_authenticated=False):
        self.node_id = node_id
        self.local_endpoints = set(local_endpoints)
        self.max_payload_bytes = max_payload_bytes
        self.max_storage_bytes = max_storage_bytes
        self.local_lifetime_override_ms = local_lifetime_override_ms
        self.require_authenticated = require_authenticated
        self.seen: set[tuple] = set()
        self.held: dict[tuple, ToyBundle] = {}
        self.storage_used = 0

    def _validate(self, bundle: ToyBundle) -> None:
        if not isinstance(bundle, ToyBundle) or bundle.version != 7:
            raise InputError("BPv7 ToyBundle이 아닙니다")
        if not isinstance(bundle.payload, bytes) or len(bundle.payload) > self.max_payload_bytes:
            raise InputError("payload 크기 제한 위반")
        integers = (bundle.creation_time_ms, bundle.sequence_number, bundle.lifetime_ms)
        if any(type(value) is not int or value < 0 for value in integers):
            raise InputError("primary 숫자 필드가 잘못되었습니다")
        ext = bundle.extensions
        if ext.bundle_age_ms is not None and (type(ext.bundle_age_ms) is not int or ext.bundle_age_ms < 0):
            raise InputError("Bundle Age 범위 위반")
        if bundle.creation_time_ms == 0 and ext.bundle_age_ms is None:
            raise InputError("creation time=0이면 Bundle Age가 필요합니다")
        if (ext.hop_limit is None) != (ext.hop_count is None):
            raise InputError("hop limit과 count는 함께 있어야 합니다")
        if ext.hop_limit is not None:
            if not 1 <= ext.hop_limit <= 255 or type(ext.hop_count) is not int or ext.hop_count < 0:
                raise InputError("Hop Count 범위 위반")
        if bundle.source == self.node_id and ext.previous_node is not None:
            raise InputError("로컬 source bundle에는 Previous Node가 없어야 합니다")

    def _age(self, bundle: ToyBundle, now_ms: int, clock_accurate: bool) -> int:
        if bundle.creation_time_ms != 0 and clock_accurate:
            if now_ms < bundle.creation_time_ms:
                raise InputError("정확한 시계가 생성 시각보다 과거입니다")
            return now_ms - bundle.creation_time_ms
        if bundle.extensions.bundle_age_ms is None:
            raise InputError("age를 판정할 수 없습니다")
        return bundle.extensions.bundle_age_ms

    def _delete(self, reason: str, events: list[str]) -> ProcessResult:
        events.append(f"DELETED: {reason}; retention constraints 제거")
        return ProcessResult(State.DELETED, reason, None, tuple(events))

    def _forward(self, bundle: ToyBundle, residence_ms: int, events: list[str]) -> ProcessResult:
        ext = bundle.extensions
        age = ext.bundle_age_ms + residence_ms if ext.bundle_age_ms is not None else None
        count = ext.hop_count + 1 if ext.hop_count is not None else None
        updated = replace(bundle, extensions=Extensions(self.node_id, age, ext.hop_limit, count))
        events.append(f"extensions 갱신: previous={self.node_id}, age={age}, hop={count}")
        if count is not None and count > ext.hop_limit:
            return self._delete("Hop limit exceeded", events)
        events.append("FORWARDED: CLA에 전달(모의) 후 Forward pending 제거")
        return ProcessResult(State.FORWARDED, "forwarded", updated, tuple(events))

    def receive(self, bundle: ToyBundle, *, now_ms: int, clock_accurate: bool = True,
                route_available: bool = True, residence_ms: int = 0,
                authenticated: bool = False) -> ProcessResult:
        events = ["RECEIVED: Dispatch pending 추가"]
        try:
            self._validate(bundle)
            if self.require_authenticated and not authenticated:
                return self._delete("local policy: unauthenticated", events)
            if bundle.identity in self.seen:
                return self._delete("local policy: duplicate identity", events)
            age = self._age(bundle, now_ms, clock_accurate)
        except (InputError, TypeError) as exc:
            return self._delete(f"Block unintelligible: {exc}", events)
        events.append(f"VALIDATED: age={age}ms")
        self.seen.add(bundle.identity)
        effective = bundle.lifetime_ms
        reason = "Lifetime expired"
        if self.local_lifetime_override_ms is not None and self.local_lifetime_override_ms < effective:
            effective = self.local_lifetime_override_ms
            reason = "Traffic pared"
        if age > effective:
            return self._delete(reason, events)
        if bundle.extensions.hop_count is not None and bundle.extensions.hop_count > bundle.extensions.hop_limit:
            return self._delete("Hop limit exceeded", events)
        events.append("DISPATCHED: Dispatch pending 제거")
        if bundle.destination in self.local_endpoints:
            events.append("DELIVERED: ADU를 application agent에 제시")
            return ProcessResult(State.DELIVERED, "local endpoint", bundle, tuple(events))
        if route_available:
            return self._forward(bundle, residence_ms, events)
        if self.storage_used + len(bundle.payload) > self.max_storage_bytes:
            return self._delete("Depleted storage", events)
        self.held[bundle.identity] = bundle
        self.storage_used += len(bundle.payload)
        events.append("FORWARD_PENDING: 다음 contact까지 persistent storage에 보관(모의)")
        return ProcessResult(State.FORWARD_PENDING, "no timely contact", bundle, tuple(events))

## 3. 정상 경로: local delivery와 세 relay

첫 bundle은 목적지에서 전달된다. 두 번째 bundle은 hop limit=2로 세 relay를 지나려 한다. count=2는 아직 초과가 아니지만 세 번째 forwarding에서 3이 되어 삭제된다.

In [4]:
destination = ToyNode("dtn://mars", {"dtn://mars/inbox"})
delivered = destination.receive(parsed, now_ms=50_000, clock_accurate=False)
assert delivered.state is State.DELIVERED
print("[local delivery]", *delivered.events, sep="\n  " )

current = parsed
for index, residence in enumerate((50, 25, 10), start=1):
    relay = ToyNode(f"dtn://relay-{index}")
    result = relay.receive(current, now_ms=50_000, clock_accurate=False, residence_ms=residence)
    print(f"\n[relay {index}] {result.state.value}: {result.reason}")
    print(*result.events, sep="  - " )
    if result.bundle is None:
        break
    current = result.bundle

assert result.state is State.DELETED and result.reason == "Hop limit exceeded"

[local delivery]
  RECEIVED: Dispatch pending 추가
  VALIDATED: age=100ms
  DISPATCHED: Dispatch pending 제거
  DELIVERED: ADU를 application agent에 제시

[relay 1] forwarded: forwarded
RECEIVED: Dispatch pending 추가  - VALIDATED: age=100ms  - DISPATCHED: Dispatch pending 제거  - extensions 갱신: previous=dtn://relay-1, age=150, hop=1  - FORWARDED: CLA에 전달(모의) 후 Forward pending 제거

[relay 2] forwarded: forwarded
RECEIVED: Dispatch pending 추가  - VALIDATED: age=150ms  - DISPATCHED: Dispatch pending 제거  - extensions 갱신: previous=dtn://relay-2, age=175, hop=2  - FORWARDED: CLA에 전달(모의) 후 Forward pending 제거

[relay 3] deleted: Hop limit exceeded
RECEIVED: Dispatch pending 추가  - VALIDATED: age=175ms  - DISPATCHED: Dispatch pending 제거  - extensions 갱신: previous=dtn://relay-3, age=185, hop=3  - DELETED: Hop limit exceeded; retention constraints 제거


## 4. malformed input과 자원 고갈 테스트

실패 입력을 예외 메시지로 끝내지 않고, 어느 trust boundary에서 왜 거부했는지 확인한다. `Depleted storage`는 RFC status reason code이지만 여기서 정확한 status report bundle을 만들지는 않는다.

In [5]:
def expect_input_error(label, raw, **limits):
    try:
        parse_untrusted(raw, **limits)
    except InputError as exc:
        print(f"{label}: 거부 - {exc}")
    else:
        raise AssertionError(f"{label}은 거부되어야 합니다")

duplicate_age = dict(raw_bundle)
duplicate_age["extension_blocks"] = raw_bundle["extension_blocks"] + [{"type": 7, "value": 101}]
expect_input_error("중복 Bundle Age", duplicate_age)

oversized = dict(raw_bundle, payload=b"x" * 33)
expect_input_error("과대 payload", oversized, max_payload_bytes=32)

bool_lifetime = dict(raw_bundle, lifetime_ms=True)
expect_input_error("bool을 정수로 위장", bool_lifetime)

unknown_extension = dict(raw_bundle)
unknown_extension["extension_blocks"] = [{"type": 250, "value": b"private"}]
expect_input_error("미지원 extension", unknown_extension)

중복 Bundle Age: 거부 - extension type 7 중복
과대 payload: 거부 - payload 타입 또는 크기 제한 위반
bool을 정수로 위장: 거부 - lifetime은 음이 아닌 int여야 합니다
미지원 extension: 거부 - 이 toy가 처리할 수 없는 extension type: 250


In [6]:
limited_node = ToyNode("dtn://small-relay", max_storage_bytes=20)
first_waiting = replace(parsed, sequence_number=10, payload=b"A" * 12)
second_waiting = replace(parsed, sequence_number=11, payload=b"B" * 12)
held = limited_node.receive(first_waiting, now_ms=0, clock_accurate=False, route_available=False)
rejected = limited_node.receive(second_waiting, now_ms=0, clock_accurate=False, route_available=False)
assert held.state is State.FORWARD_PENDING
assert rejected.state is State.DELETED and rejected.reason == "Depleted storage"
assert limited_node.storage_used == 12
print("첫 bundle:", held.state.value, "/ 저장량", limited_node.storage_used)
print("두 번째 bundle:", rejected.state.value, "/", rejected.reason)

# asserted lifetime은 그대로 두고 local effective lifetime만 짧게 적용한다.
paring_node = ToyNode("dtn://defender", local_lifetime_override_ms=50)
long_lived = replace(parsed, sequence_number=12, lifetime_ms=10**12,
                     extensions=replace(parsed.extensions, bundle_age_ms=51))
pared = paring_node.receive(long_lived, now_ms=0, clock_accurate=False)
assert pared.reason == "Traffic pared" and long_lived.lifetime_ms == 10**12
print("수명 override 결과:", pared.state.value, pared.reason)

첫 bundle: forward_pending / 저장량 12
두 번째 bundle: deleted / Depleted storage
수명 override 결과: deleted Traffic pared


## 5. 보안 경계: CRC ≠ authenticity

CRC는 전송 오류는 잡을 수 있지만 공격자는 payload와 CRC를 함께 다시 계산할 수 있다. RFC 9171 §8은 BP 계층 보안이 필요할 때 BPSec(RFC 9172)를 사용하도록 한다. 아래 HMAC은 ‘키를 가진 주체만 tag를 만들 수 있다’는 차이만 보이는 로컬 데모이며, BIB canonicalization, security context, key management를 구현하지 않는다.

In [7]:
def demo_tag(bundle: ToyBundle, key: bytes) -> bytes:
    # 이 ad-hoc 표현은 BPv7/BPSec wire format이 아니다.
    message = (bundle.source.encode() + b"\0" + bundle.destination.encode() +
               b"\0" + bundle.payload)
    return hmac.new(key, message, hashlib.sha256).digest()

demo_key = b"lab-only-secret"
tag = demo_tag(parsed, demo_key)
tampered = replace(parsed, payload=parsed.payload + b"!")
assert hmac.compare_digest(tag, demo_tag(parsed, demo_key))
assert not hmac.compare_digest(tag, demo_tag(tampered, demo_key))
print("원본 tag 검증: True")
print("변조 payload tag 검증: False")

strict_node = ToyNode("dtn://strict", require_authenticated=True)
unauthenticated = strict_node.receive(parsed, now_ms=0, clock_accurate=False, authenticated=False)
assert unauthenticated.state is State.DELETED
print("인증 필수 local policy:", unauthenticated.reason)

원본 tag 검증: True
변조 payload tag 검증: False
인증 필수 local policy: local policy: unauthenticated


## 6. 전체 자동 검수와 예상 결과

성공 시 마지막 줄에 `심화 실습 검수 통과`가 출력된다. 이 검수는 toy의 동작 계약만 확인하며 RFC 적합성 시험 suite가 아니다.

In [8]:
assert delivered.state is State.DELIVERED
assert result.state is State.DELETED and result.reason == "Hop limit exceeded"
assert rejected.reason == "Depleted storage"
assert pared.reason == "Traffic pared"
assert unauthenticated.reason == "local policy: unauthenticated"

# lifetime 경계: age == lifetime은 만료가 아니고 다음 forwarding으로 진행한다.
boundary = replace(parsed, sequence_number=99, lifetime_ms=100,
                   extensions=Extensions(bundle_age_ms=100))
boundary_result = ToyNode("dtn://boundary").receive(
    boundary, now_ms=0, clock_accurate=False, route_available=True
)
assert boundary_result.state is State.FORWARDED

# creation time 0인데 age block이 없으면 Block unintelligible로 삭제한다.
missing_age = replace(parsed, sequence_number=100, extensions=Extensions())
missing_age_result = ToyNode("dtn://validator").receive(
    missing_age, now_ms=0, clock_accurate=False
)
assert missing_age_result.state is State.DELETED
assert "Block unintelligible" in missing_age_result.reason
print("심화 실습 검수 통과")

심화 실습 검수 통과


## 연습 문제와 실제 구현으로 가는 길

1. `held` bundle을 contact가 열릴 때 꺼내 forward하고 저장량을 감소시키는 메서드를 추가하라.
2. 알 수 없는 extension에 block processing flags를 넣고 RFC 9171 §5.6의 report/delete/discard/continue 분기를 구현하라.
3. fragment identity와 bounded reassembly cache를 02의 `Reassembler`로 결합하라. timeout, 최대 fragment 수, 발신자별 quota를 함께 설계하라.
4. 상태 전이마다 status report 요청 flag가 켜진 경우 관리 record를 enqueue하되, 증폭 DoS를 막는 rate limit을 추가하라.
5. 실제 구현을 목표로 한다면 RFC 8949 결정적 CBOR, RFC 9171 CDDL/errata, RFC 9172 BPSec, RFC 9174 TCPCLv4를 함께 검토하고 공식 interoperability vector로 검증하라.

### 보안 점검표

- decode 전에 전체 입력 바이트와 중첩·항목 수에 상한이 있는가?
- reassembly storage가 발신자별·전체별로 제한되고 lifetime override가 가능한가?
- node ID와 convergence-layer peer mapping을 인증하는가?
- primary block이 cleartext라는 사실을 traffic-analysis 모델에 반영했는가?
- CRC 성공을 발신자 인증 성공으로 오해하지 않는가?